# Week 47, Day 6: Forge: Week 47 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", 10 / 50)
print("2.", round(1 / np.sqrt(25), 2))
print("3.", round(2 * np.pi * 40 * 0.005, 2), "rad")

### Exercise: Part A: fitting and testing the period law

Simulate the network for tau_I = 3, 5, 8 and 16 ms (600 ms each), find each peak frequency, fit 1/f = t0 + k tau_I with np.linalg.lstsq, and compare the prediction at 12 ms with a fifth simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ping(tauI=8.0, T=600.0, dt=0.1, seed=0, NE=400, NI=100, drive=2.2, noise=0.05):
    g = np.random.default_rng(seed); p = 0.2
    W_EI = (g.random((NI, NE)) < p) * 0.15; W_IE = (g.random((NE, NI)) < p) * 1.2; W_II = (g.random((NI, NI)) < p) * 0.4
    vE, vI = g.random(NE), g.random(NI); xE, xIE, xII = np.zeros(NI), np.zeros(NE), np.zeros(NI)
    steps = int(T / dt); rateE = np.zeros(steps)
    for k in range(steps):
        vE += dt / 10 * (-vE + drive - xIE) + noise * np.sqrt(dt) * g.normal(size=NE)
        vI += dt / 10 * (-vI + 0.5 + xE - xII) + noise * np.sqrt(dt) * g.normal(size=NI)
        fE, fI = vE >= 1, vI >= 1; vE[fE] = 0; vI[fI] = 0
        xE += -dt * xE / 2.0 + W_EI @ fE; xIE += -dt * xIE / tauI + W_IE @ fI; xII += -dt * xII / tauI + W_II @ fI
        rateE[k] = fE.mean()
    return rateE

def peak_freq(rate, dt=0.1):
    x = rate[2000:] - rate[2000:].mean(); f = np.fft.rfftfreq(len(x), dt * 1e-3); P = np.abs(np.fft.rfft(x)) ** 2
    m = (f > 10) & (f < 200)
    return f[m][P[m].argmax()]

taus = np.array([3.0, 5.0, 8.0, 16.0])
freqs = np.array([peak_freq(ping(t)) for t in taus])
coef = ...
pred = ...
actual = peak_freq(ping(12.0))
print(f"frequencies {freqs} Hz; t0 = {1000 * coef[0]:.1f} ms, k = {1000 * coef[1]:.2f} ms per ms; predicted at 12 ms {pred:.0f} Hz, simulated {actual:.0f} Hz")
tt = np.linspace(2, 20, 50)
plt.plot(taus, freqs, "o", label="simulated"); plt.plot(tt, 1 / (coef[0] + coef[1] * tt), label="fitted law"); plt.plot(12, actual, "s", label="test")
plt.xlabel("tau_I (ms)"); plt.ylabel("frequency (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "the fitted slope k is positive"
    assert coef[1] > 0
def test_predict():
    "the predicted frequency at tau_I = 12 ms is within 15% of the simulated one"
    assert abs(pred - actual) / actual < 0.15

test_slope()
test_predict()
print("All checks passed")

### Exercise: Part B: a one-feature sleep scorer

Write slow_fraction(X): the power of the mean-subtracted epoch between 0.1 and 2 Hz divided by its power between 0.1 and 40 Hz. Compute it for all 40 epochs and classify those above 0.5 as sleep.

In [ ]:
f_ = lambda u: 1 / (1 + np.exp(-(u - 3) / 0.5))
def run(I, b=2.0, T=10.0, dt=0.002, seed=0):
    g = np.random.default_rng(seed); n = int(T / dt); x = a = 0.0; X = np.zeros(n)
    nz = 0.15 * np.sqrt(dt / 0.01) * g.normal(size=n)
    for k in range(n):
        x += dt / 0.01 * (-x + f_(6 * x + I - a)) + nz[k]; a += dt / 0.5 * (-a + b * x); X[k] = x
    return X

def slow_fraction(X, dt=0.002):
    ...

g = np.random.default_rng(1)
labels = g.integers(0, 2, 40)                 # 0 sleep-like, 1 waking-like
feats = np.array([slow_fraction(run(g.uniform(0.8, 1.2) if l == 0 else g.uniform(1.9, 2.3), seed=i)) for i, l in enumerate(labels)])
predicted = np.where(feats > 0.5, 0, 1)
acc = np.mean(predicted == labels)
print(f"mean slow fraction: sleep {feats[labels == 0].mean():.2f}, waking {feats[labels == 1].mean():.2f}; accuracy {acc:.3f}")
print("misclassified epochs:", np.where(predicted != labels)[0], "with features", np.round(feats[predicted != labels], 2))

In [ ]:
import numpy as np
def test_separate():
    "sleep-like epochs have a slow-power fraction above 0.6 on average, waking-like below 0.3"
    assert feats[labels == 0].mean() > 0.6 and feats[labels == 1].mean() < 0.3
def test_accuracy():
    "a threshold of 0.5 classifies at least 90% of epochs correctly"
    assert acc >= 0.9

test_separate()
test_accuracy()
print("All checks passed")